# MindScreen: Temporal Drift and Calibration Failure in NHANES-Based Depression Screening

**Reproduction and extension of:** Vu et al. (2025), *Prediction of depressive disorder using machine learning approaches: findings from the NHANES*.

## What this notebook establishes, in order

1. **Setup & Preprocessing** — builds four NHANES waves (2013-14, 2015-16, 2017-18, 2021-23) into a single feature set, faithfully reproducing Vu et al.'s clinical variable definitions (with one documented, deliberate deviation -- see the OGTT note below).
2. **Phase 0 (Lab Assay Audit)** — checks whether NHANES changed its laboratory methodology across cycles, to rule out measurement artifacts before trusting any cross-wave comparison.
3. **Phase 1** — reproduces the paper's same-wave baseline, compares four model families plus a non-ML clinical rule, and tunes XGBoost's hyperparameters. The tuned configuration becomes canonical for every phase after this one.
4. **Phase 2** — the core temporal-generalization experiment: train on the past, test on progressively more distant future waves. Produces the temporal decay curve, the single strongest piece of evidence in this notebook.
5. **Phase 3 / 3.5** — bootstrap-ensemble risk stratification, and calibration analysis (Platt scaling). Finds that the model's *rankings* degrade modestly with time, but its *calibration* breaks more severely and in a way that doesn't survive standard recalibration.
6. **Phase 4 / 4b** — SHAP explainability, both for the single production model and across separately-trained wave-specific models, to test whether *what the model relies on* is stable over time.
7. **Phase 5** — tests whether NHANES survey weighting changes model performance (it doesn't, meaningfully).
8. **Phase 6 / 7** — translates calibration failure into population-scale impact, tests it under multiple threshold-setting conventions, and builds a cheap fix (prevalence-shift correction) with its own sensitivity analysis.

**One item remains explicitly flagged:** the NHANES laboratory bridging statistics in Phase 0 are qualitative only; no specific sample sizes or correlation coefficients are quoted pending primary-source verification against CDC laboratory documentation.


In [ ]:
import os
import json
import warnings
import io
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import roc_auc_score, average_precision_score, matthews_corrcoef, roc_curve
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance
from scipy.stats import ks_2samp
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
warnings.filterwarnings('ignore')


## 1. Setup

Mounts Google Drive and defines the single config block (paths, random seed, CV folds, bootstrap count) used everywhere in this notebook -- changed in one place, not per-cell.

**Data availability and ethics statement.** NHANES data are publicly available, de-identified survey data released by the CDC's National Center for Health Statistics. This analysis uses only existing, publicly released, de-identified datasets and is exempt from institutional review board approval.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# --- Single config block ---
PROJECT_ROOT = '/content/drive/MyDrive/MindScreen'
SAS_MISSING_FLOAT = 5.397605e-79
RANDOM_STATE = 42
N_CV_SPLITS = 5
MARITAL_INCLUSION_THRESHOLD = 0.005
N_BOOTSTRAP = 500
# --- Temporal gap convention ---
# Gap = end_year(test_wave) - end_year(training_wave)
# Using end-of-cycle dates because deployment decisions are made after
# training data collection ends.
WAVE_END_YEARS = {
    '2013-2014': 2014,
    '2015-2016': 2016,
    '2017-2018': 2018,
    '2021-2023': 2023,
}

for sub in ['raw_data', 'processed_data', 'results', 'figures']:
    os.makedirs(f'{PROJECT_ROOT}/{sub}', exist_ok=True)

print("Config loaded. PROJECT_ROOT:", PROJECT_ROOT)

Mounted at /content/drive
Config loaded. PROJECT_ROOT: /content/drive/MyDrive/MindScreen


## 2. Preprocessing Utilities

**The SAS missing-value artifact.** NHANES `.xpt` files encode missing values as a specific tiny float (`5.397605e-79`) rather than a standard NaN. `clean_sas()` converts this to a proper NaN everywhere it appears in non-outcome columns.

**A deliberate decision on the depression label itself.** For the PHQ-9 items that build the `depression` label, a missing item is treated as NaN (true missingness), not imputed as 0. Since PHQ-9 has no internal skip logic, a missing value reflects incomplete interview data, not a legitimate skip. This choice was verified against the original reproduction checkpoint (N=5,372, 511 depressed, 9.51% prevalence, matching Vu et al. exactly).

In [ ]:
def clean_sas(df):
    """Replace the SAS tiny-float missing-value artifact with NaN."""
    return df.replace(SAS_MISSING_FLOAT, np.nan)

def decode_col(col):
    """Decode bytes columns to strings."""
    return col.apply(lambda x: x.decode('utf-8') if isinstance(x, bytes) else x)

def process_dpq(dpq_raw):
    """
    Clean PHQ-9 and derive the depression label (PHQ-9 sum >= 10).

    """
    dpq = dpq_raw.copy()
    phq9_cols = ['DPQ010','DPQ020','DPQ030','DPQ040','DPQ050',
                 'DPQ060','DPQ070','DPQ080','DPQ090']
    #  Replace SAS missing with NaN (not 0)
    dpq[phq9_cols] = dpq[phq9_cols].replace(SAS_MISSING_FLOAT, np.nan)
    dpq[phq9_cols] = dpq[phq9_cols].replace({7: np.nan, 9: np.nan})  # Refused / Don't know
    dpq['phq9_complete'] = dpq[phq9_cols].notna().all(axis=1)
    dpq['phq9_sum'] = dpq[phq9_cols].sum(axis=1, skipna=False)
    dpq['depression'] = (dpq['phq9_sum'] >= 10).astype('Int64')
    dpq.loc[dpq['phq9_sum'].isna(), 'depression'] = np.nan
    return dpq

print("Preprocessing utilities defined.")

Preprocessing utilities defined.


## 3. Medication Classification

**Vu et al.'s actual definitions** (verified against the paper's Methods text):
- Hypertension: SBP ≥140, DBP ≥90, **or** antihypertensive medication use.
- Diabetes: fasting glucose ≥126, HbA1c ≥6.5%, 2-hour OGTT ≥200, **or** diabetic medication use.
- Dyslipidemia: total cholesterol ≥200, triglycerides ≥150, LDL ≥140, HDL <40, **or** lipid-lowering medication use.

Medication use is an equal, non-optional criterion in all three -- not a fallback for missing labs.

**Cycle-specific medication data:** cycles H/I/J (2013-2018) use drug-class matching against the NHANES Multum Lexicon reference file (`RXQ_DRUG`). Cycle L (2021-2023) has no compatible drug-class reference available, so medication use there is taken from direct self-report (`DIQ050`/`DIQ070` for diabetes, `BPQ150` for hypertension, `BPQ101D` for lipids), merged explicitly on `SEQN`. This is a genuine cross-wave measurement-method difference (self-report vs. pharmacy-verified classification), disclosed here as a limitation rather than hidden.

**Deliberate deviation from Vu et al.: the OGTT criterion is not implemented, in any wave.** OGTT (oral glucose tolerance test) requires a dedicated fasting visit and is not part of a routine outpatient checkup -- implementing it would conflict with this project's core design constraint (screen using only data a routine checkup already collects). It also has substantially higher missingness in NHANES than fasting glucose or HbA1c. This means a small number of people with normal fasting glucose/HbA1c but abnormal post-challenge glucose ("isolated post-challenge hyperglycemia") will be missed by the diabetes flag in every wave -- a known, accepted limitation, not an inconsistency between waves.

In [ ]:
def derive_medication_flags(rxq_table, rxq_drug_table, cycle_letter=None,
                            bpq_table=None, diq_table=None):
    """Build hypertension/diabetes/lipid medication flags."""
    if cycle_letter == "L" and bpq_table is not None and diq_table is not None:
        flags = bpq_table[["SEQN"]].copy()
        flags["flag_htn_med"] = (bpq_table["BPQ150"] == 1).astype(bool)
        flags["flag_lipid_med"] = (bpq_table["BPQ101D"] == 1).astype(bool)

        dm_flags = diq_table[["SEQN"]].copy()
        dm_flags["flag_dm_med"] = (
            (diq_table["DIQ050"] == 1) | (diq_table["DIQ070"] == 1)
        ).astype(bool)

        flags = pd.merge(flags, dm_flags, on="SEQN", how="left")
        flags["flag_dm_med"] = flags["flag_dm_med"].fillna(False).astype(bool)
        return flags

    # Cycles H/I/J: drug-class matching using the master RXQ_DRUG file
    # NOTE: RXQ_DRUG is a SINGLE master file (1988-2020), not per-cycle files.
    # The CDC documentation confirms this:
    # "RXQ_DRUG contains therapeutic drug class information on all drugs
    #  reported by NHANES participants from 1988-1994 through 2017-March 2020"
    rxq_active = rxq_table[rxq_table['RXDUSE'] == 1].copy()
    drug_class_cols = ['RXDDRGID', 'RXDDCN1B', 'RXDDCN2B', 'RXDDCN3B', 'RXDDCN4B']
    rxq_drug_trim = rxq_drug_table[drug_class_cols].copy()
    for col in drug_class_cols[1:]:
        rxq_drug_trim[col] = decode_col(rxq_drug_trim[col])
    rxq_active = pd.merge(rxq_active, rxq_drug_trim, on='RXDDRGID', how='left')

    antihypertensive_classes = {
        'ANTIHYPERTENSIVE COMBINATIONS', 'ANGIOTENSIN CONVERTING ENZYME (ACE) INHIBITORS',
        'ANGIOTENSIN II INHIBITORS', 'CALCIUM CHANNEL BLOCKING AGENTS', 'DIURETICS',
    }
    antidiabetic_classes = {'ANTIDIABETIC AGENTS'}
    antihyperlipidemic_classes = {'ANTIHYPERLIPIDEMIC AGENTS'}
    level2_cols = ['RXDDCN1B', 'RXDDCN2B', 'RXDDCN3B', 'RXDDCN4B']

    def matches_class(row, class_set):
        return any(row[col] in class_set for col in level2_cols)

    rxq_active['flag_htn_med'] = rxq_active.apply(matches_class, axis=1, class_set=antihypertensive_classes)
    rxq_active['flag_dm_med'] = rxq_active.apply(matches_class, axis=1, class_set=antidiabetic_classes)
    rxq_active['flag_lipid_med'] = rxq_active.apply(matches_class, axis=1, class_set=antihyperlipidemic_classes)

    rxq_person = rxq_active.groupby('SEQN')[['flag_htn_med', 'flag_dm_med', 'flag_lipid_med']].max().reset_index()
    return rxq_person

print("derive_medication_flags defined.")

derive_medication_flags defined.


In [ ]:
def fetch_with_retry(url, max_retries=3):
    """Fetch URL with retry logic for transient CDC server errors."""
    for attempt in range(max_retries):
        try:
            response = requests.get(url, timeout=30)
            response.raise_for_status()
            # Verify it's actually an XPORT file, not an HTML error page
            content = response.content
            if content[:8] == b'HEADER R' or content[:6] == b'SAS   ':
                return content
            # Check if it looks like HTML (error page)
            if b'<html' in content[:100].lower() or b'<!doctype' in content[:100].lower():
                raise ValueError(f"Server returned HTML error page for {url}")
            return content
        except Exception as e:
            if attempt == max_retries - 1:
                raise e
            print(f"  Retry {attempt + 1}/{max_retries} for {url}...")
            import time
            time.sleep(2)


## 4. Wave-Loading Pipeline

Downloads, merges, and derives clinical variables for one NHANES wave at a time. Handles the structural differences between cycle L and earlier cycles: 3-reading oscillometric blood pressure (`BPXO`) instead of 2-reading manual (`BPX`), `LBXTLG` instead of `LBXTR` for triglycerides, and frequency-coded physical activity questions instead of yes/no. `fetch_with_retry()` guards against transient CDC server errors and against silently treating an HTML error page as valid data.

An `on_bp_meds` column is also carried through as a diagnostic check (not a modeling feature) -- useful for spot-checking the `hypertension` composite flag against a single, simple predictor.

In [ ]:
def load_and_merge_wave(year, cycle_letter, rxq_drug_table, alq_cols=('ALQ101', 'ALQ110')):
    """Download, clean, merge, and derive clinical variables for one NHANES wave."""
    base_url = f"https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/{year}/DataFiles/"
    is_L = (cycle_letter == "L")

    def fetch(fname):
        content = fetch_with_retry(base_url + fname + f"_{cycle_letter}.xpt")
        return pd.read_sas(io.BytesIO(content), format='xport')

    demo = fetch("DEMO")
    dpq = process_dpq(fetch("DPQ"))
    merged = pd.merge(demo, dpq, on='SEQN', how='inner')

    bmx = clean_sas(fetch("BMX")[['SEQN', 'BMXBMI']])

    if is_L:
        bpx_raw = clean_sas(fetch("BPXO")[['SEQN', 'BPXOSY1', 'BPXOSY2', 'BPXOSY3',
                                             'BPXODI1', 'BPXODI2', 'BPXODI3']])
        bpx_raw['SBP'] = bpx_raw[['BPXOSY1', 'BPXOSY2', 'BPXOSY3']].mean(axis=1)
        bpx_raw['DBP'] = bpx_raw[['BPXODI1', 'BPXODI2', 'BPXODI3']].mean(axis=1)
    else:
        bpx_raw = clean_sas(fetch("BPX")[['SEQN', 'BPXSY1', 'BPXSY2', 'BPXDI1', 'BPXDI2']])
        bpx_raw['SBP'] = bpx_raw[['BPXSY1', 'BPXSY2']].mean(axis=1)
        bpx_raw['DBP'] = bpx_raw[['BPXDI1', 'BPXDI2']].mean(axis=1)
    bpx = bpx_raw[['SEQN', 'SBP', 'DBP']]

    if is_L:
        bpq_diag = clean_sas(fetch("BPQ")[['SEQN', 'BPQ150']])
        bpq_diag['on_bp_meds'] = bpq_diag['BPQ150'].map({1: 1, 2: 0})
    else:
        bpq_diag = clean_sas(fetch("BPQ")[['SEQN', 'BPQ050A']])
        bpq_diag['on_bp_meds'] = bpq_diag['BPQ050A'].map({1: 1, 2: 0})
    bpq_diag = bpq_diag[['SEQN', 'on_bp_meds']]

    smq = clean_sas(fetch("SMQ")[['SEQN', 'SMQ020', 'SMQ040']])

    alq_raw = fetch("ALQ")
    # ALQ column harmonization: 2013-2016 uses ALQ101 (annual drinking);
    # 2017-2020 uses ALQ111 (lifetime ever-drinker), renamed here for code consistency.
    # The semantic mismatch is flagged in prepare_wave() and in Limitations.
    alq = clean_sas(alq_raw[['SEQN', alq_cols[0], alq_cols[1]]]).rename(
        columns={alq_cols[0]: 'ALQ101', alq_cols[1]: 'ALQ110'}
    )

    if is_L:
        paq_raw = clean_sas(fetch("PAQ")[['SEQN', 'PAD810Q', 'PAD790Q']])
        for old, new in [('PAD810Q', 'PAQ605'), ('PAD790Q', 'PAQ620')]:
            paq_raw[new] = pd.to_numeric(paq_raw[old], errors='coerce')
            paq_raw[new] = np.where(paq_raw[new] > 0, 1, np.where(paq_raw[new] == 0, 2, np.nan))
        paq = paq_raw[['SEQN', 'PAQ605', 'PAQ620']]
    else:
        paq = clean_sas(fetch("PAQ")[['SEQN', 'PAQ605', 'PAQ620']])

    diq = clean_sas(fetch("DIQ")[['SEQN', 'DIQ010']])

    glu = clean_sas(fetch("GLU")[['SEQN', 'LBXGLU']])
    ghb = clean_sas(fetch("GHB")[['SEQN', 'LBXGH']])
    tchol = clean_sas(fetch("TCHOL")[['SEQN', 'LBXTC']])

    if is_L:
        trigly = clean_sas(fetch("TRIGLY")[['SEQN', 'LBXTLG', 'LBDLDL']]).rename(columns={'LBXTLG': 'LBXTR'})
    else:
        trigly = clean_sas(fetch("TRIGLY")[['SEQN', 'LBXTR', 'LBDLDL']])

    hdl = clean_sas(fetch("HDL")[['SEQN', 'LBDHDD']])
    biopro = clean_sas(fetch("BIOPRO")[['SEQN', 'LBXSCR']])

    for df in [bmx, bpx, bpq_diag, smq, alq, paq, diq, glu, ghb, tchol, trigly, hdl, biopro]:
        merged = pd.merge(merged, df, on='SEQN', how='left')

    # Medication flags
    if is_L:
        bpq_full = clean_sas(fetch("BPQ")[['SEQN', 'BPQ150', 'BPQ101D']])
        diq_full = clean_sas(fetch("DIQ")[['SEQN', 'DIQ050', 'DIQ070']])
        rxq_person = derive_medication_flags(None, None, cycle_letter="L",
                                              bpq_table=bpq_full, diq_table=diq_full)
    else:
        rxq = fetch("RXQ_RX")
        rxq_person = derive_medication_flags(rxq, rxq_drug_table)

    merged = pd.merge(merged, rxq_person, on='SEQN', how='left')
    for col in ['flag_htn_med', 'flag_dm_med', 'flag_lipid_med']:
        merged[col] = merged[col].infer_objects(copy=False).fillna(False).astype(bool)

    # --- Derived clinical variables, NaN-aware ---
    bp_missing = merged[['SBP','DBP']].isna().all(axis=1)
    merged['hypertension'] = np.where(
        bp_missing & (~merged['flag_htn_med']), np.nan,
        ((merged['SBP'] >= 140) | (merged['DBP'] >= 90) | (merged['flag_htn_med'])).astype(float)
    )

    glu_missing = merged[['LBXGLU','LBXGH']].isna().all(axis=1)
    merged['diabetes'] = np.where(
        glu_missing & (~merged['flag_dm_med']), np.nan,
        ((merged['LBXGLU'] >= 126) | (merged['LBXGH'] >= 6.5) | (merged['flag_dm_med'])).astype(float)
    )

    # NOTE: LDL threshold kept at 140 to match original specification.
    lipid_missing = merged[['LBXTC','LBXTR','LBDLDL','LBDHDD']].isna().all(axis=1)
    merged['dyslipidemia'] = np.where(
        lipid_missing & (~merged['flag_lipid_med']), np.nan,
        ((merged['LBXTC'] >= 200) | (merged['LBXTR'] >= 150) |
         (merged['LBDLDL'] >= 140) | (merged['LBDHDD'] < 40) |
         (merged['flag_lipid_med'])).astype(float)
    )

    merged['eGFR'] = (
        175  * (merged['LBXSCR'] ** -1.154) * (merged['RIDAGEYR'] ** -0.203) *
        np.where(merged['RIAGENDR'] == 2, 0.742, 1.0)
    )

    return merged

print("load_and_merge_wave defined.")


load_and_merge_wave defined.
